# **2.1 Data Preparation**

The objective of this section is to transform the raw monthly trading data into a format that can be used efficiently for model fitting and backtesting.

The raw bin files are stored in long format, with one row per stock, date and intraday time bin. For price impact modelling, it is more convenient to work with matrices where each row corresponds to one stock-day and each column corresponds to one intraday time bin.

The baseline project setup uses one month as the in-sample training period and the following month as the out-of-sample testing period. The same set of 20 stocks is used in both periods.


Let

$$
q_{i,d,t}
$$

denote the signed traded volume of stock \(i\) on date \(d\) during intraday bin \(t\), and let

$$
P_{i,d,t}
$$

denote the mid price at the end of the same bin.

The aim is to construct two matrices:

$$
Q_{(i,d),t} = q_{i,d,t},
$$

and

$$
P_{(i,d),t} = P_{i,d,t}.
$$

Here, the row index \((i,d)\) represents one stock-day, while the columns represent intraday time bins.

We use January 2019 as the in-sample period and February 2019 as the out-of-sample period.

The in-sample data is used for stock selection and model fitting. The out-of-sample data is only used later to evaluate the fitted model.

This avoids look-ahead bias: the stock universe is selected using only the training month, not the testing month.

The 20-stock universe is selected by liquidity. For each stock \(i\), we compute total absolute traded volume in the training month:

$$
V_i = \sum_{d \in \text{train}} \sum_t |q_{i,d,t}|.
$$

The selected universe is then

$$
\mathcal{S}_{20}
=
\text{top 20 stocks ranked by } V_i.
$$

The same stock set $\mathcal{S}_{20}$ is then used for both the training month and the testing month.

The bin files contain both signed volume and mid prices.

For signed trading volume, we use the column `trade`:

$$
Q_{(i,d),t} = \text{trade}_{i,d,t}.
$$

For prices, we use the column `midEnd`:

$$
P_{(i,d),t} = \text{midEnd}_{i,d,t}.
$$

We convert the raw long data into wide stock-day matrices:

- rows: `(stock, date)`;
- columns: `time`;
- values: either `trade` or `midEnd`.

Missing values are treated differently for trades and prices.

For traded volume, missing values are filled with zero:

$$
q_{i,d,t} = 0
$$

when there is no recorded trade in that bin.

For prices, missing values are forward-filled and backward-filled across time because the absence of a new price observation does not mean that the price is zero. The last available mid price is carried forward.

The output of the data preparation step is:

$$
Q^{\text{train}}, \quad P^{\text{train}}, \quad Q^{\text{test}}, \quad P^{\text{test}}.
$$

These matrices are saved as intermediate results and will be used in the next section to fit price impact models.

This completes the baseline data preparation step.

In [2]:
import os
import pandas as pd

data_dir = "data/"

bin_sample_path = f"{data_dir}binSamples/"
fill_sample_path = f"{data_dir}fillSamples/"

print(os.listdir(bin_sample_path))
print(os.listdir(fill_sample_path))

['bin201901.csv', 'bin201902.csv', 'bin201903.csv', 'bin201904.csv', 'bin201905.csv', 'bin201906.csv', 'bin201907.csv', 'bin201908.csv', 'bin201909.csv', 'bin201910.csv', 'bin201911.csv', 'bin201912.csv']
['fills201901.csv', 'fills201902.csv', 'fills201903.csv', 'fills201904.csv', 'fills201905.csv', 'fills201906.csv', 'fills201907.csv', 'fills201908.csv', 'fills201909.csv', 'fills201910.csv', 'fills201911.csv', 'fills201912.csv']


In [16]:
# Reading data

file_path_bin = f"{bin_sample_path}bin201901.csv"
bin_df = pd.read_csv(file_path_bin)

file_path_fill = f"{fill_sample_path}fills201901.csv"
fill_df = pd.read_csv(file_path_fill)


In [9]:
# January 2019 = in-sample
# February 2019 = out-of-sample

year = 2019
train_month = 1
test_month = 2

In [10]:
month = "%02d" % train_month
train_bin_df = pd.read_csv(bin_sample_path + f"bin{year}{month}.csv")

month = "%02d" % test_month
test_bin_df = pd.read_csv(bin_sample_path + f"bin{year}{month}.csv")

In [12]:
# Choosing the 20 most liquid stocks

stock_volume = (
    train_bin_df
    .groupby("stock")["trade"]
    .apply(lambda x: x.abs().sum())
    .sort_values(ascending=False)
)

stocks_20 = list(stock_volume.head(20).index)

In [22]:
# Saving the stocks

result_path = "data/"
os.makedirs(result_path, exist_ok=True)

pd.DataFrame({"stock": stocks_20}).to_csv(
    result_path + "stocks_20_201901.csv",
    index=False
)

In [13]:
# Filtering train and test to the same 20 stocks

train_bin_df = train_bin_df.loc[train_bin_df["stock"].isin(stocks_20)].copy()
test_bin_df = test_bin_df.loc[test_bin_df["stock"].isin(stocks_20)].copy()

In [17]:
train_traded_volume_df = train_bin_df[["stock", "date", "trade", "time"]].pivot(
    index=["stock", "date"],
    columns="time",
    values="trade"
).fillna(0)

train_px_df = train_bin_df[["stock", "date", "midEnd", "time"]].pivot(
    index=["stock", "date"],
    columns="time",
    values="midEnd"
).ffill(axis="columns").bfill(axis="columns")


test_traded_volume_df = test_bin_df[["stock", "date", "trade", "time"]].pivot(
    index=["stock", "date"],
    columns="time",
    values="trade"
).fillna(0)

test_px_df = test_bin_df[["stock", "date", "midEnd", "time"]].pivot(
    index=["stock", "date"],
    columns="time",
    values="midEnd"
).ffill(axis="columns").bfill(axis="columns")

In [ ]:
# Saving the files 

train_traded_volume_df.reset_index().to_csv(
    result_path + "train_traded_volume_201901_20.csv",
    index=False
)

train_px_df.reset_index().to_csv(
    result_path + "train_px_201901_20.csv",
    index=False
)

test_traded_volume_df.reset_index().to_csv(
    result_path + "test_traded_volume_201902_20.csv",
    index=False
)

test_px_df.reset_index().to_csv(
    result_path + "test_px_201902_20.csv",
    index=False
)